**Note:** This notebook is designed for **Google Colab**.

If you see the Colab logo <span style='vertical-align:bottom;'><img src='https://colab.research.google.com/img/colab_favicon_256px.png' width='40' alt='Colab logo'></span> in the top-left corner, you're all set! Please **proceed to Section 1**.

If you don't see the logo (e.g., you are on GitHub), please click the button below to open it in the correct environment:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mparrott-at-wiris/aimodelshare/blob/master/notebooks/justice_and_equity_advance_notebook_en.ipynb)

# **Advanced Justice & Equity Challenge: Build & Submit Custom Models**

Welcome to the **Advanced Pathway** of the Ethics at Play (Ètica en Joc) Justice Challenge. 

**Who is this for?** 
This notebook is designed for participants with Python experience (e.g., Scikit-Learn, TensorFlow, PyTorch). Instead of using the gamified apps, you will build, train, and submit your own machine learning models directly to the competition leaderboard.

**The Goal:** 
Train a model to predict recidivism risk (the likelihood of re-offending) using the COMPAS dataset, while balancing accuracy and fairness.

## 🚀 **Quick Start Guide**

To participate in the challenge, complete these 5 steps:

1.  **Install Libraries:** Run the setup cell to install `aimodelshare`.
2.  **Get the Data:** Run the data loading cell to retrieve the COMPAS dataset.
3.  **Train Your Model:** Use the provided Scikit-Learn Pipeline example or write your own custom training code.
4.  **Connect:** Link this notebook to the Justice Challenge Leaderboard.
5.  **Submit:** Send your predictions to the leaderboard to see your score.

**Ready? Click the ▶ Play Button on the first cell below to get started.**

---
# **Step 1: Installation**

We need to install the `aimodelshare` library to connect to the competition backend.

In [ ]:
# Install the aimodelshare library
print("Installing required libraries...")
!pip install aimodelshare --upgrade -q --no-warn-script-location > /dev/null 2>&1
print("✅ Installation complete!")

---
# **Step 2: Load Data**

We will use the **COMPAS** dataset, which is the standard dataset used for this challenge. 

In this step, we use specific functions to process the raw data (calculating length of stay, grouping charges, etc.) and create a stratified train/test split that matches the leaderboard's evaluation protocol.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# --- CONFIGURATION ---
MAX_ROWS_TEST = 4000                              # reproduce original X_TEST sampling
COMPAS_URL = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"

ALL_NUMERIC_COLS = ["juv_fel_count", "juv_misd_count", "juv_other_count", "days_b_screening_arrest", "age", "length_of_stay", "priors_count"]
ALL_CATEGORICAL_COLS = ["race", "sex", "c_charge_degree", "c_charge_desc"]
ALL_FEATURES = ALL_NUMERIC_COLS + ALL_CATEGORICAL_COLS

# --- DATA PREP FUNCTIONS ---
def load_and_prepare(df: pd.DataFrame, max_rows: int | None):
    try:
        df = df.copy()
        df['c_jail_in'] = pd.to_datetime(df['c_jail_in'])
        df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
        df['length_of_stay'] = (df['c_jail_out'] - df['c_jail_in']).dt.total_seconds() / (24 * 60 * 60)
    except Exception:
        df = df.copy()
        df['length_of_stay'] = np.nan

    if max_rows is not None and df.shape[0] > max_rows:
        df = df.sample(n=max_rows, random_state=42)

    # Process Charge Description: Keep top 50, map rest to "OTHER"
    if "c_charge_desc" in df.columns:
        top_charges = df["c_charge_desc"].value_counts().head(50).index
        df["c_charge_desc"] = df["c_charge_desc"].apply(lambda x: x if pd.notna(x) and x in top_charges else "OTHER")

    # Ensure all features exist
    for col in ALL_FEATURES:
        if col not in df.columns:
            df[col] = np.nan

    X = df[ALL_FEATURES].copy()
    y = df["two_year_recid"].copy()
    return X, y

def load_original_test_split():
    print(f"Downloading data from {COMPAS_URL}...")
    df = pd.read_csv(COMPAS_URL)
    X, y = load_and_prepare(df, max_rows=MAX_ROWS_TEST)
    
    # Stratified split to match evaluation backend
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    return X_train_raw, X_test_raw, y_train, y_test

# --- EXECUTION ---
# Load data and assign to variables for the next steps
X_train, X_test, y_train, y_test = load_original_test_split()

print("✅ Data loaded and processed using advanced configuration!")
print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")
print("\nFirst 5 rows of training data:")
X_train.head()

---
# **Step 3: Train Model with Pipeline**

We will use a **Scikit-Learn Pipeline** to streamline preprocessing and modeling. 

This pipeline will:
1.  **Impute Missing Values** (Fill NaNs with median for numbers, most frequent for categories).
2.  **One-Hot Encode** categorical columns (Race, Sex, Charge Degree, Charge Desc).
3.  **Scale** numerical columns (Age, Priors, Length of Stay, etc.).
4.  **Train** a Logistic Regression classifier.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Define feature groups using constants from Step 2
# Numerical features will be imputed and scaled
numeric_features = ALL_NUMERIC_COLS

# Categorical features will be imputed and One-Hot Encoded
categorical_features = ALL_CATEGORICAL_COLS

# 2. Define Transformers with Imputation
# Numeric: Impute missing values with the median, then scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical: Impute missing values with the most frequent value, then one-hot encode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# 3. Create Preprocessor using the transformers
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 4. Create Pipeline (Preprocessor + Model)
# You can replace LogisticRegression with any other sklearn model (e.g., RandomForestClassifier)
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# 5. Train the pipeline
pipeline.fit(X_train, y_train)

# 6. Generate predictions on the test set
predictions = pipeline.predict(X_test)

# 7. Evaluate accuracy
accuracy = accuracy_score(y_test, predictions)
print(f"✅ Model Trained! Accuracy: {accuracy:.2%}")

---
# **Step 4: Connect to the Leaderboard**

This step connects your notebook to the specific backend for the Justice & Equity Challenge. 

*Note: You will be prompted to enter a username and password. If you don't have one, check with your instructor or the challenge website.*

In [ ]:
from aimodelshare.aws import set_credentials
from aimodelshare.playground import Competition

# The specific Model Playground URL for the Justice Challenge
my_playground_url = "https://cf3wdpkg0d.execute-api.us-east-1.amazonaws.com/prod/m"

# Set your credentials (pop-up will appear)
set_credentials(apiurl=my_playground_url)

# Connect to the competition
playground = Competition(my_playground_url)

---
# **Step 5: Submit & Check Results**

Submit your predictions to the leaderboard.

In [ ]:
# 1. Submit your predictions
# Note: We pass None for model and preprocessor because we are only submitting predictions for evaluation
playground.submit_model(
    model=None,
    preprocessor=None,
    prediction_submission=predictions,
    input_dict={
        "description": "Logistic Regression with Sklearn Pipeline", 
        "tags": "sklearn, logistic_regression, advanced_pathway, pipeline"
    }
)

print("✅ Predictions submitted successfully!")

# 2. Check the leaderboard
print("Loading leaderboard...")
leaderboard = playground.get_leaderboard()
playground.stylize_leaderboard(leaderboard)